# Trading Bot – Full Pipeline

**Anleitung:**
1. Accelerator auf **GPU T4 x2** stellen (Settings → Accelerator)
2. Dataset **busersteven/trading-raw-data** hinzufügen (Add data → Your datasets)
3. Alles ausführen: **Run All** oder **Save Version → Save & Run All (Commit)**

Die Pipeline:
- Lädt `scripts/kaggle_full_run.py` von GitHub (main); in der **Python-Zelle** steht `HORIZONS = [11, 15]` (anpassbar; überschreibt den Stand von main)
- Klont danach das Repo für Trainings-/Backtest-Code
- Installiert Dependencies, kopiert Parquet-Daten
- Single-Horizon-Training (Walk-Forward) und Backtest
- Packt Artefakte nach `/kaggle/working/kaggle_artifacts.tar.gz`

In [ ]:
# Aktuellen Pipeline-Code von GitHub laden und ausführen
import re
import subprocess
import sys
import time

# Single-Horizon: welche Tage in SCHRITT 20 trainiert werden — hier einstellen.
# (GitHub main kann noch [4,7] haben; diese Zeile überschreibt SH_HORIZONS vor exec.)
HORIZONS = [11, 15]

# Cache-Buster: raw.githubusercontent.com cached bis zu 5 Min
cache_bust = int(time.time())
url = f"https://raw.githubusercontent.com/stevenlangeshops/trading/main/scripts/kaggle_full_run.py?cb={cache_bust}"
r = subprocess.run(
    ["wget", "-q", "-O", "/kaggle/working/kaggle_full_run.py", url],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(r.stdout or "download ok")

path = "/kaggle/working/kaggle_full_run.py"
with open(path, encoding="utf-8") as f:
    code = f.read()
code, n_sub = re.subn(
    r"^(\s*SH_HORIZONS\s*=\s*)\[[^\]]+\]",
    lambda m: m.group(1) + repr(HORIZONS),
    code,
    count=1,
    flags=re.MULTILINE,
)
if n_sub == 0:
    print("[WARN] SH_HORIZONS konnte nicht ersetzt werden — prüfe kaggle_full_run.py auf GitHub.")
else:
    print(f"SH_HORIZONS -> {HORIZONS} (vor exec gesetzt)")

# Modul-Cache leeren (verhindert Stale-Code bei Re-Runs)
for mod in list(sys.modules.keys()):
    if mod.startswith(("strategy", "models", "features")):
        del sys.modules[mod]

exec(code)